In [1]:
# ===============================
# Helcim Take Home Technical Challenge
# Interview 4: SQL Preparation
# ===============================

# Author: Shelly Resurreccion
# Purpose: Practice writing SQL queries that may come up based on the dataset provided.

---
### 1.0 Set Up

In [2]:
# ===============================
# Imports:
# ===============================
# --- Standard library ---
import sys
import os

# --- Data handling ---
import sqlite3
import pandas as pd

In [3]:
# ===============================
# Functions:
# ===============================
# N/A

In [4]:
# ===============================
# Load Data:
# ===============================
# Notes: The dataset was encoded in Latin-1 rather than UTF-8, so the encoding was explicitly specified when loading the CSV.
df = pd.read_csv(
    "../data/US_Accidents_March23.csv",
    encoding="latin1"
)

conn = sqlite3.connect(":memory:")
US_Accidents = df.to_sql("US_Accidents", conn, index=False, if_exists="replace")

In [5]:
df.columns

Index(['Unnamed: 0', 'ID', 'Source', 'Severity', 'Start_Time', 'End_Time',
       'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)',
       'Description', 'Street', 'City', 'County', 'State', 'Zipcode',
       'Country', 'Timezone', 'Airport_Code', 'Weather_Timestamp',
       'Temperature_Range(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)',
       'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)',
       'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing',
       'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station',
       'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop',
       'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight',
       'Astronomical_Twilight'],
      dtype='object')

In [6]:
# Question 1
# Top 5 states with the most severe accidents

query = """
SELECT
    State,
    COUNT(*) as Accident_Count
FROM US_Accidents
WHERE Severity >= 3
GROUP BY State
ORDER BY Accident_Count DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,State,Accident_Count
0,VA,1153
1,NC,800
2,GA,749
3,PA,681
4,CO,469


In [7]:
# Question 2
# Accidents by hour + severity

query = """
SELECT
    strftime('%H', Start_Time) AS Hour,
    Severity,
    COUNT(*) AS Accident_Count
FROM US_Accidents
GROUP BY Hour, Severity
ORDER BY Hour, Severity;
"""

pd.read_sql(query, conn)

,Hour,Severity,Accident_Count
0,00,2,4374
1,00,4,234
2,01,2,3582
3,01,4,163
4,02,2,3491
5,02,4,163
6,03,2,3185
7,03,4,146
8,04,2,3603
9,04,4,155


In [8]:
# Question 3
# Weather conditions associated with highest severity

query = """
SELECT
    Weather_Condition,
    Severity,
    COUNT(*) AS Accident_Count
FROM US_Accidents
WHERE Severity >= 3
GROUP BY Weather_Condition, Severity
ORDER BY Accident_Count DESC;
"""

pd.read_sql(query, conn)

,Weather_Condition,Severity,Accident_Count
0,Fair,4,2682
1,Cloudy,4,1639
2,Mostly Cloudy,4,592
3,Light Rain,4,431
4,Partly Cloudy,4,364
5,Light Snow,4,360
6,None,4,271
7,Fog,4,208
8,Rain,4,83
9,Wintry Mix,4,55


In [9]:
# Question 4
# Rush hour vs off-peak accident comparison

query = """
SELECT
    CASE
        WHEN CAST(strftime('%H', Start_Time) AS Integer) BETWEEN 7 and 9
            OR CAST(strftime('%H', Start_Time) AS Integer) BETWEEN 16 and 18
        THEN 'Rush Hour'
        ELSE 'Off Peak'
    END AS Time_Bucket,
    COUNT(*) AS Accident_Count
FROM US_Accidents
GROUP BY Time_Bucket;
"""

pd.read_sql(query, conn)

,Time_Bucket,Accident_Count
0,Off Peak,154245
1,Rush Hour,92388


In [ ]:
# Question 5
# Month-over-month accident trend

query = """
SELECT
    Start_Time,
    strftime('%Y', Start_Time) AS Year,
    strftime('%m', Start_Time) AS Month,
    COUNT(*) AS Accident_Count
FROM US_Accidents
GROUP BY Year, Month;
"""

pd.read_sql(query, conn)

,Start_Time,Year,Month,Accident_Count
0,2023-01-16 17:00:36.000000000,2023,01,160914
1,2023-02-27 20:55:00.000000000,2023,02,55532
2,2023-03-31 17:09:16.000000000,2023,03,30187


In [11]:
# Question 6
# Accidents in California in January 2023

query = """
SELECT 
    COUNT(*)
FROM US_Accidents
WHERE State = 'CA'
    AND strftime('%Y-%m-%d', Start_Time) >= '2023-01-01'
    AND strftime('%Y-%m-%d', Start_Time) < '2023-01-31';
"""

pd.read_sql(query, conn)

,COUNT(*)
0,34618


In [12]:
# Question 7
# Accidents by day of the week

query = """
SELECT strftime('%w', Start_Time) AS Day_Of_Week,
       COUNT(*) AS Accident_Count
FROM US_Accidents
GROUP BY Day_Of_Week
ORDER BY Day_Of_Week;
"""

pd.read_sql(query, conn)

,Day_Of_Week,Accident_Count
0,0,27727
1,1,36863
2,2,41421
3,3,39547
4,4,38174
5,5,39873
6,6,23028


In [13]:
# Question 8
# States with more than 10,000 severe accidents

query = """
SELECT 
    State,
    COUNT(*) AS Accident_Count
FROM US_Accidents
WHERE Severity >= 3
GROUP BY State
HAVING COUNT(*) >= 100
ORDER BY Accident_Count DESC;
"""

pd.read_sql(query, conn)

,State,Accident_Count
0,VA,1153
1,NC,800
2,GA,749
3,PA,681
4,CO,469
5,FL,313
6,MI,265
7,CA,234
8,NY,207
9,IL,194


In [14]:
# Question 9
# Average severity by weather condition (excluding NULLs)

query = """
SELECT 
    Weather_Condition,
    AVG(Severity) as Avg_Severity
FROM US_Accidents
WHERE Weather_Condition IS NOT NULL
GROUP BY Weather_Condition
ORDER BY Avg_Severity DESC;
"""

pd.read_sql(query, conn)

,Weather_Condition,Avg_Severity
0,Light Rain Shower,2.200000
1,Blowing Snow / Windy,2.166667
2,Blowing Dust / Windy,2.133333
3,Light Freezing Rain,2.120805
4,Heavy Snow / Windy,2.101266
...,...,...
75,Freezing Drizzle,2.000000
76,Drizzle and Fog,2.000000
77,Drizzle / Windy,2.000000
78,Blowing Snow,2.000000


In [15]:
# Question 10
# Top 3 states by number of accidents per year

# State
# COUNT(*) AS Accident_Count
# strftime('%Y', Start_Time) AS Year

# Rank States by the Accident_Count so I need a ranking mechanism too
# We could use a nested function or a CTE

query = """
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER(PARTITION BY State ORDER BY Accident_Count DESC) AS rn
    FROM (
        SELECT 
            State, 
            strftime('%m', Start_Time) AS Month, 
            COUNT(*) AS Accident_Count
        FROM US_Accidents
        GROUP BY State, Month
    )
)
WHERE rn <= 3
ORDER BY Accident_Count DESC
LIMIT 3;
"""

pd.read_sql(query, conn)

,State,Month,Accident_Count,rn
0,CA,01,35867,1
1,CA,02,23360,2
2,FL,01,20689,1


In [16]:
# Question 11
# Show all states in the top 25% of accidents

# Using a CTE 

query = """
WITH state_accidents AS (
    SELECT State,
           COUNT(*) AS Accident_Count
    FROM US_Accidents
    GROUP BY State
),

state_ranked AS (
    SELECT *,
           PERCENT_RANK() OVER(ORDER BY Accident_Count ASC) AS pct_rank
    FROM state_accidents
)

SELECT 
    State, 
    Accident_Count, 
    pct_rank
FROM state_ranked
WHERE pct_rank >= 0.75
ORDER BY Accident_Count DESC;
"""

pd.read_sql(query, conn)

,State,Accident_Count,pct_rank
0,CA,74559,1.000000
1,FL,22243,0.977273
2,TX,13824,0.954545
3,VA,11426,0.931818
4,NY,10820,0.909091
5,PA,10605,0.886364
6,NC,9133,0.863636
7,MN,8949,0.840909
8,SC,8534,0.818182
9,GA,8457,0.795455


In [22]:
# Question 12
# How does each city compare to the median number of accidents?

query = """
WITH city_counts AS (
    SELECT City,
           COUNT(*) AS accident_count
    FROM US_Accidents
    GROUP BY City
),
avg_count AS (
    SELECT AVG(accident_count) AS avg_accidents
    FROM city_counts
)
SELECT c.City,
       c.accident_count,
       a.avg_accidents,
       c.accident_count - a.avg_accidents AS diff_from_avg
FROM city_counts c
CROSS JOIN avg_count a
ORDER BY diff_from_avg DESC;
"""

pd.read_sql(query, conn)

,City,accident_count,avg_accidents,diff_from_avg
0,Los Angeles,5880,37.115576,5842.884424
1,Miami,5238,37.115576,5200.884424
2,Dallas,3261,37.115576,3223.884424
3,Atlanta,3182,37.115576,3144.884424
4,San Diego,2759,37.115576,2721.884424
...,...,...,...,...
6640,Zenia,1,37.115576,-36.115576
6641,Zieglerville,1,37.115576,-36.115576
6642,Zortman,1,37.115576,-36.115576
6643,Zuni,1,37.115576,-36.115576


In [ ]:
# Question 13
# Median in SQLite

query = """
WITH ordered AS (
        SELECT 
                [Distance(mi)],
                ROW_NUMBER() OVER (ORDER BY [Distance(mi)]) AS rn,
                COUNT(*) OVER () AS cnt
        FROM US_Accidents
        )
SELECT [Distance(mi)] as median
FROM ordered
WHERE rn = (cnt / 2);
"""

#123317 is the mid point

pd.read_sql(query, conn)

,median
0,0.292


In [38]:
# Question 14
# Distance between two points (classic interview question)

query = """
SELECT
    ID,
    Start_Lat,
    Start_Lng,
    End_Lat,
    End_Lng,
    [Distance(mi)],
    6371 * 2 * ASIN(
        SQRT(
            POWER(SIN(RADIANS(Start_Lat - End_Lat) / 2), 2) +
            COS(RADIANS(End_Lat)) * COS(RADIANS(Start_Lat)) *
            POWER(SIN(RADIANS(Start_Lng - End_Lng) / 2), 2)
        )
    ) AS distance_km
FROM US_Accidents;
"""

#123317 is the mid point

pd.read_sql(query, conn)

,ID,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),distance_km
0,A-3650461,45.676472,-94.174568,45.666976,-94.176184,0.661,1.063346
1,A-3650462,41.391812,-75.467365,41.398370,-75.484203,0.983,1.582557
2,A-3650463,40.850278,-73.946140,40.846945,-73.933651,0.692,1.113939
3,A-3650464,33.714992,-84.300188,33.714532,-84.266758,1.922,3.092470
4,A-3650465,33.927671,-118.266385,33.929608,-118.343065,4.398,7.077939
...,...,...,...,...,...,...,...
246628,A-5464675,29.697432,-95.423674,29.697499,-95.425230,0.094,0.150478
246629,A-5464680,33.869027,-84.366612,33.853174,-84.370037,1.113,1.790917
246630,A-5464682,39.603673,-86.069712,39.609702,-86.072118,0.436,0.701366
246631,A-5464712,32.917276,-96.717765,32.916783,-96.717758,0.034,0.054823


In [43]:
# Question 15
# Compute the duration of the accident

query = """
SELECT 
    State,
    Start_Time,
    End_Time,
    (strftime('%s', End_Time) - strftime('%s', Start_Time)) AS duration_seconds
FROM US_Accidents
ORDER BY duration_seconds DESC;
"""

pd.read_sql(query, conn)

,State,Start_Time,End_Time,duration_seconds
0,OR,2023-01-16 17:00:36.000000000,2023-03-27 14:42:31.000000000,6039715
1,OR,2023-01-16 17:00:36,2023-02-26 15:29:52,3536956
2,OR,2023-02-09 17:59:21.000000000,2023-03-21 01:30:07.000000000,3396646
3,OR,2023-01-16 17:00:36,2023-02-21 12:11:59,3093083
4,CA,2023-01-08 15:55:00,2023-02-13 00:01:09,3053169
...,...,...,...,...
246628,CA,2023-02-04 19:27:30,2023-02-04 19:33:00,330
246629,TX,2023-03-24 19:55:30,2023-03-24 20:00:30,300
246630,CA,2023-01-30 17:16:30,2023-01-30 17:21:30,300
246631,UT,2023-01-17 21:44:30.000000000,2023-01-17 21:49:00.000000000,270


In [45]:
# Question 16
# Which state had the longest duration of accidents?

query = """
WITH duration AS (
    SELECT 
        State,
        Start_Time,
        End_Time,
        (strftime('%s', End_Time) - strftime('%s', Start_Time)) AS duration_seconds
    FROM US_Accidents
)

SELECT
    State,
    SUM(duration_seconds) AS total_duration_seconds
FROM duration
GROUP BY State
ORDER BY total_duration_seconds DESC;
"""

pd.read_sql(query, conn)

,State,total_duration_seconds
0,CA,599566842
1,FL,187763268
2,TX,136257949
3,NY,91785772
4,OR,82001018
5,PA,71675557
6,NC,68998097
7,VA,65451748
8,AZ,59739860
9,SC,54524884
